# Head 2: Cause Classifier Training

Training a 6-class mental health **cause** classifier (No reason, Bias or abuse, Jobs and careers, Medication, Relationship, Alienation) on a MentalBERT backbone using the CAMS dataset.

This notebook is one component of a two-head classification pipeline. Head 1 (Condition classifier on Mukherjee + Dreaddit) is trained separately, and mirrors this notebook's structure.

**Input data**: The raw `CAMS.csv` file (`text`, `category`, `explanation` columns). Unlike Head 1, this dataset comes as a single file, so this notebook cleans it and performs its own stratified train/val/test split rather than reading pre-split CSVs.

**Output**: A trained checkpoint, label mapping, and test evaluation results.

In [ ]:
import os
import sys
import json
import random
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score
)
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("SECTION 1: ENVIRONMENT CHECK")

if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU detected: {gpu_name}")
    print(f"VRAM: {gpu_mem:.1f} GB")
else:
    device = torch.device("cpu")
    print("WARNING: No GPU detected.")
    print("Enable GPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

# Load HF token from Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if HF_TOKEN:
        print("HF_TOKEN loaded from environment variable.")
    else:
        print(f"WARNING: Could not load HF_TOKEN: {e}")
        print("MentalBERT requires authentication. Add HF_TOKEN as a Kaggle Secret.")

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

## Section 2: Load, Clean, and Split Data

Load the raw `CAMS.csv` file. Unlike Head 1, there is no pre-split train/val/test set and no `source_dataset` column (CAMS is a single source), so this section:

1. Drops rows with a missing `text` or missing `category` (label)
2. Casts `category` to integer class ids
3. Performs a stratified 80/10/10 train/val/test split, seeded for reproducibility

Row counts and class distributions are printed for each resulting split to confirm the data arrived correctly before any training begins.

In [ ]:
print("SECTION 2: LOAD, CLEAN, AND SPLIT DATA")

DATA_DIR = "/kaggle/input/datasets/daltonkhatri/head2-cams"
CAMS_PATH = os.path.join(DATA_DIR, "CAMS.csv")

raw_df = pd.read_csv(CAMS_PATH)
print(f"Raw rows loaded: {len(raw_df):,d}")
print(f"Columns: {list(raw_df.columns)}")

# Drop rows with missing text or missing category label
n_before = len(raw_df)
clean_df = raw_df.dropna(subset=["text", "category"]).copy()
n_dropped = n_before - len(clean_df)
print(f"\nDropped {n_dropped:,d} rows with missing text or category")
print(f"Remaining rows: {len(clean_df):,d}")

# Category arrives as float (e.g. 5.0); cast to int class id
clean_df["category"] = clean_df["category"].astype(int)

# Rename to match the naming convention used in the Head 1 pipeline
clean_df = clean_df.rename(columns={"text": "cleaned_text"})

print("\nOverall category distribution (raw ids):")
counts = clean_df["category"].value_counts().sort_index()
for cat_id, count in counts.items():
    pct = 100 * count / len(clean_df)
    print(f"    {cat_id}  {count:>6,d}  ({pct:5.1f}%)")

# Stratified 80/10/10 split
train_df, temp_df = train_test_split(
    clean_df, test_size=0.20, stratify=clean_df["category"], random_state=RANDOM_SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["category"], random_state=RANDOM_SEED
)

for name, df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"\n{name}: {len(df):,d} rows")
    counts = df["category"].value_counts().sort_index()
    for cat_id, count in counts.items():
        pct = 100 * count / len(df)
        print(f"    {cat_id}  {count:>6,d}  ({pct:5.1f}%)")

## Section 3: Label Encoding

Map the six CAMS cause categories to integer indices. CAMS ships with the label already encoded as an integer (0 through 5); this section attaches the human-readable class names in the standard CAMS ordering and saves the mapping to disk so it can be reused later without guessing the order.

In [ ]:
print("SECTION 3: LABEL ENCODING")

LABEL_TO_ID = {
    "No reason": 0,
    "Bias or abuse": 1,
    "Jobs and careers": 2,
    "Medication": 3,
    "Relationship": 4,
    "Alienation": 5,
}
ID_TO_LABEL = {v: k for k, v in LABEL_TO_ID.items()}

print("\nLabel mapping:")
for label, idx in LABEL_TO_ID.items():
    print(f"  {label} -> {idx}")

# The CSV's "category" column already holds these integer ids
for df in [train_df, val_df, test_df]:
    df["label"] = df["category"]

# Verify every label falls within the expected range
for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    n_invalid = (~df["label"].isin(ID_TO_LABEL.keys())).sum()
    if n_invalid > 0:
        print(f"  WARNING: {name} has {n_invalid} labels outside the expected 0-5 range")
    else:
        print(f"  {name}: all labels valid")

# Save the mapping
mapping_path = os.path.join(CHECKPOINT_DIR, "head2_label_mapping.json")
with open(mapping_path, "w") as f:
    json.dump({str(k): v for k, v in ID_TO_LABEL.items()}, f, indent=2)
print(f"\nLabel mapping saved to {mapping_path}")

## Section 4: Dataset and DataLoader

A PyTorch Dataset that tokenizes text on the fly using the MentalBERT tokenizer, same as Head 1 (max length 256, same tokenizer).

CAMS is a single-source dataset, so the domain-balanced sampler used in Head 1 is not needed here. The training DataLoader uses standard random shuffling instead.

In [ ]:
print("SECTION 4: DATASET AND DATALOADER")

MAX_LENGTH = 256
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32


class CauseDataset(Dataset):
    """PyTorch Dataset that tokenizes cleaned text and returns model inputs."""

    def __init__(self, dataframe, tokenizer, max_length=256):
        self.texts = dataframe["cleaned_text"].tolist()
        self.labels = dataframe["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx]) if self.texts[idx] is not None else ""
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


# Load tokenizer (same MentalBERT tokenizer as Head 1)
print("Loading MentalBERT tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    "mental/mental-bert-base-uncased", token=HF_TOKEN
)
print(f"Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")

# Create datasets
train_dataset = CauseDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = CauseDataset(val_df, tokenizer, MAX_LENGTH)
test_dataset = CauseDataset(test_df, tokenizer, MAX_LENGTH)

print(f"Train dataset: {len(train_dataset):,d} samples")
print(f"Val dataset:   {len(val_dataset):,d} samples")
print(f"Test dataset:  {len(test_dataset):,d} samples")

# Create data loaders (plain shuffle for train; no domain balancing needed)
train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

## Section 5: Model

Load MentalBERT as the backbone and add a classification head on top — the same architecture pattern as Head 1: a dropout layer followed by a linear layer mapping from MentalBERT's 768-dimensional hidden state to the number of classes. The only difference from Head 1 is `num_classes=6` instead of 5. The `[CLS]` token's pooled representation is used as the input to the classification head.

Per the pipeline plan, Head 2 starts from a **fresh MentalBERT** — it does not load the Head 1 checkpoint. The two heads are trained and validated independently before being combined in Phase 4.

In [ ]:
print("SECTION 5: MODEL")

class CauseClassifier(nn.Module):
    """MentalBERT backbone with a dropout + linear classification head."""

    def __init__(self, backbone, num_classes=6, dropout_rate=0.1):
        super().__init__()
        self.backbone = backbone
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(768, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(
            input_ids=input_ids, attention_mask=attention_mask
        )
        pooled = outputs.pooler_output
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        return logits


# Load a fresh MentalBERT backbone
print("Loading MentalBERT backbone...")
backbone = AutoModel.from_pretrained(
    "mental/mental-bert-base-uncased", token=HF_TOKEN
)
print("MentalBERT loaded successfully.")

# Build the classifier
model = CauseClassifier(backbone, num_classes=6, dropout_rate=0.1)
model = model.to(device)

# Print parameter counts
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,d}")
print(f"Trainable parameters: {trainable_params:,d}")

## Section 6: Loss Function

Use weighted cross-entropy to handle class imbalance, same approach as Head 1. The weights are computed using sklearn's `compute_class_weight` with strategy "balanced", which assigns higher weight to underrepresented classes (Bias or abuse is the smallest class in CAMS at roughly 7% of rows).

In [ ]:
print("SECTION 6: LOSS FUNCTION")

# Compute class weights from training label distribution
train_labels_np = train_df["label"].values
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(sorted(LABEL_TO_ID.values())),
    y=train_labels_np,
)

print("\nComputed class weights:")
for idx, weight in enumerate(class_weights):
    label_name = ID_TO_LABEL[idx]
    count = (train_labels_np == idx).sum()
    print(f"  {label_name:20s} (n={count:>6,d}): weight = {weight:.4f}")

weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights_tensor)
print(f"\nWeighted CrossEntropyLoss created on {device}")

# Section 6.5: Training Tracker Initialization

In [ ]:
history = {
    "epoch": [],
    "train_loss": [],
    "val_macro_f1": [],
    "val_accuracy": [],
    "val_no_reason_f1": [],
    "val_bias_or_abuse_f1": [],
    "val_jobs_and_careers_f1": [],
    "val_medication_f1": [],
    "val_relationship_f1": [],
    "val_alienation_f1": [],
}
print("Training history tracker initialized.")

## Section 7: Training Loop

Train for up to 10 epochs with AdamW optimizer and linear warmup over the first 10% of total training steps, same hyperparameters as Head 1. The best model checkpoint (by validation macro F1) is saved after each improving epoch.

Training stops early if validation macro F1 does not improve for 3 consecutive epochs.

"Bias or abuse" F1 is monitored separately every epoch because it is the most underrepresented class. A warning is printed if it drops below 0.40 at any point.

In [ ]:
# Label names in index order, used throughout evaluation
LABEL_NAMES = [ID_TO_LABEL[i] for i in range(len(ID_TO_LABEL))]


def evaluate_model(model, dataloader, device):
    """Run inference on a dataloader and return predictions and true labels."""
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return np.array(all_preds), np.array(all_labels)


def compute_metrics(preds, labels):
    """Compute macro F1, per-class metrics, confusion matrix, and accuracy."""
    macro_f1 = f1_score(labels, preds, average="macro")
    accuracy = accuracy_score(labels, preds)
    report_dict = classification_report(
        labels, preds, target_names=LABEL_NAMES,
        output_dict=True, zero_division=0
    )
    report_str = classification_report(
        labels, preds, target_names=LABEL_NAMES, zero_division=0
    )
    cm = confusion_matrix(labels, preds)

    return {
        "macro_f1": macro_f1,
        "accuracy": accuracy,
        "report_dict": report_dict,
        "report_str": report_str,
        "confusion_matrix": cm,
    }


def print_epoch_metrics(epoch, train_loss, metrics):
    """Print a formatted summary of one epoch's results."""
    print(f"\n  Epoch {epoch} Summary:")
    print(f"    Train Loss:    {train_loss:.4f}")
    print(f"    Val Macro F1:  {metrics['macro_f1']:.4f}")
    print(f"    Val Accuracy:  {metrics['accuracy']:.4f}")
    print(f"\n    Per-class results:")
    print(f"    {'Class':20s} {'F1':>8s} {'Precision':>10s} {'Recall':>8s}")
    print(f"    {'-' * 48}")
    for label_name in LABEL_NAMES:
        cls = metrics["report_dict"][label_name]
        print(
            f"    {label_name:20s} {cls['f1-score']:8.4f} "
            f"{cls['precision']:10.4f} {cls['recall']:8.4f}"
        )

    # record history
    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_macro_f1"].append(metrics["macro_f1"])
    history["val_accuracy"].append(metrics["accuracy"])
    for cls in LABEL_NAMES:
        key = "val_" + cls.lower().replace(" ", "_") + "_f1"
        history[key].append(metrics["report_dict"][cls]["f1-score"])

    # Bias or abuse specific warning (most underrepresented class)
    bias_f1 = metrics["report_dict"]["Bias or abuse"]["f1-score"]
    if bias_f1 < 0.40:
        print(
            f"\n    WARNING: Bias or abuse F1 ({bias_f1:.4f}) is below 0.40. "
            f"The model is struggling with the most underrepresented class."
        )

In [ ]:
print("SECTION 7: TRAINING LOOP")

# Hyperparameters (same starting point as Head 1)
LEARNING_RATE = 2e-5
MAX_EPOCHS = 10
WARMUP_PROPORTION = 0.1
PATIENCE = 3

# Optimizer
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

# Scheduler with linear warmup
total_steps = len(train_loader) * MAX_EPOCHS
warmup_steps = int(total_steps * WARMUP_PROPORTION)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"Optimizer: AdamW (lr={LEARNING_RATE}, weight_decay=0.01)")
print(f"Total training steps: {total_steps:,d}")
print(f"Warmup steps: {warmup_steps:,d} ({WARMUP_PROPORTION * 100:.0f}%)")
print(f"Max epochs: {MAX_EPOCHS}")
print(f"Early stopping patience: {PATIENCE}")
print(f"Gradient clipping: max_norm=1.0")

best_macro_f1 = 0.0
patience_counter = 0
best_epoch = 0
checkpoint_path = os.path.join(CHECKPOINT_DIR, "best_head2_checkpoint.pt")

for epoch in range(1, MAX_EPOCHS + 1):
    print(f"\n{'=' * 60}")
    print(f"EPOCH {epoch}/{MAX_EPOCHS}")
    print(f"{'=' * 60}")

    # Training phase
    model.train()
    total_loss = 0.0
    n_batches = 0

    progress = tqdm(train_loader, desc=f"Training epoch {epoch}", leave=True)
    for batch in progress:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        n_batches += 1
        progress.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_train_loss = total_loss / n_batches

    # Validation phase
    val_preds, val_labels_arr = evaluate_model(model, val_loader, device)
    val_metrics = compute_metrics(val_preds, val_labels_arr)

    # Print epoch results
    print_epoch_metrics(epoch, avg_train_loss, val_metrics)

    # Check for new best
    current_f1 = val_metrics["macro_f1"]
    if current_f1 > best_macro_f1:
        best_macro_f1 = current_f1
        best_epoch = epoch
        patience_counter = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_macro_f1": best_macro_f1,
                "label_mapping": ID_TO_LABEL,
            },
            checkpoint_path,
        )
        print(f"\n    NEW BEST checkpoint saved (macro F1: {best_macro_f1:.4f})")
    else:
        patience_counter += 1
        print(
            f"\n    No improvement. Best: {best_macro_f1:.4f} at epoch {best_epoch}. "
            f"Patience: {patience_counter}/{PATIENCE}"
        )

    if patience_counter >= PATIENCE:
        print(
            f"\n  EARLY STOPPING triggered after {PATIENCE} epochs "
            f"without improvement."
        )
        print(f"  Best validation macro F1: {best_macro_f1:.4f} at epoch {best_epoch}")
        break

print(f"\n{'=' * 60}")
print(f"TRAINING COMPLETE")
print(f"Best epoch: {best_epoch}, Best val macro F1: {best_macro_f1:.4f}")
print(f"Checkpoint saved to: {checkpoint_path}")
print(f"{'=' * 60}")

## Section 8: Test Set Evaluation

Load the best saved checkpoint and evaluate on the held-out test set, which was not seen during training or validation.

Results are compared against reference baselines: random guessing (~16.7% macro F1 for 6 classes) and typical fine-tuned BERT performance on similar tasks.

In [ ]:
print("SECTION 8: TEST SET EVALUATION")

# Load best checkpoint
print(f"Loading best checkpoint from epoch {best_epoch}...")
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Checkpoint loaded. Val macro F1 was: {checkpoint['val_macro_f1']:.4f}")

# Run on test set
test_preds, test_true = evaluate_model(model, test_loader, device)
test_metrics = compute_metrics(test_preds, test_true)

# Print results
print(f"\nTest Set Results:")
print(f"  Macro F1:  {test_metrics['macro_f1']:.4f}")
print(f"  Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"\nClassification Report:")
print(test_metrics["report_str"])
print(f"Confusion Matrix:")
print(f"  Rows = true labels, Columns = predicted labels")
print(f"  Label order: {LABEL_NAMES}")
print(test_metrics["confusion_matrix"])

# Baseline comparison
print(f"\nBaseline Comparison:")
print(f"  Random baseline (6 classes):   ~0.1670 macro F1")
print(f"  Strong fine-tuned BERT:        ~0.55 to 0.75 macro F1 (CAMS is a harder, noisier task than Head 1)")
print(f"  This model:                     {test_metrics['macro_f1']:.4f} macro F1")

if test_metrics["macro_f1"] < 0.40:
    print(f"\n  NOTE: Test macro F1 ({test_metrics['macro_f1']:.4f}) is below 0.40.")
    print(f"  This is below the expected range for a fine-tuned model.")
    print(f"  Possible causes to investigate:")
    print(f"    1. Label encoding mismatch between training and evaluation")
    print(f"    2. Data leakage or contamination in the splits")
    print(f"    3. Learning rate too high or too low")
    print(f"    4. Not enough training epochs before early stopping")

# Save results to file
results_path = os.path.join(CHECKPOINT_DIR, "head2_test_results.txt")
with open(results_path, "w") as f:
    f.write("HEAD 2 (CAUSE CLASSIFIER) TEST SET RESULTS\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"Best epoch: {best_epoch}\n")
    f.write(f"Val macro F1 at best epoch: {checkpoint['val_macro_f1']:.4f}\n\n")
    f.write(f"Test Macro F1:  {test_metrics['macro_f1']:.4f}\n")
    f.write(f"Test Accuracy:  {test_metrics['accuracy']:.4f}\n\n")
    f.write("Classification Report:\n")
    f.write(test_metrics["report_str"])
    f.write(f"\nConfusion Matrix:\n")
    f.write(f"Label order: {LABEL_NAMES}\n")
    for row in test_metrics["confusion_matrix"]:
        f.write("  " + "  ".join(f"{v:>5d}" for v in row) + "\n")
    f.write("\n")

print(f"\nResults saved to {results_path}")

## Note: No Confound Check for Head 2

Head 1's Section 10 checked whether errors clustered by `source_dataset` (Mukherjee vs. Dreaddit), since that model was trained on two combined sources. CAMS is a single-source dataset with no `source_dataset` column, so that check does not apply here and is intentionally omitted. If CAMS is later combined with another cause-labeled dataset, the same confound-check pattern from Head 1 can be reused directly.

## Summary

All steps complete. Output files in `/kaggle/working/checkpoints/`:

| File | Contents |
|------|----------|
| `best_head2_checkpoint.pt` | Best model weights, optimizer state, epoch, label mapping |
| `head2_label_mapping.json` | Integer to CAMS category name mapping |
| `head2_test_results.txt` | Full test evaluation results with confusion matrix |
| `head2_training_curves.png` | Training loss, val F1/accuracy, per-class F1, confusion matrix plots |

Download these files from the Kaggle **Output** tab for use in Phase 3 (validating both heads) and Phase 4 (joint two-head training) of the pipeline.

# Section 8.5: Training Curves

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

epochs = history["epoch"]

fig = plt.figure(figsize=(16, 10))
fig.suptitle("Head 2 — Cause Classifier Training Results", fontsize=14, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(2, 2, hspace=0.45, wspace=0.35)

# Plot 1: Training Loss
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(epochs, history["train_loss"], "o-", color="#e05c5c", linewidth=2, markersize=5)
ax1.set_title("Training Loss per Epoch")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_xticks(epochs)
ax1.grid(True, alpha=0.3)

# Plot 2: Val Macro F1 + Accuracy
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(epochs, history["val_macro_f1"], "o-", color="#4c8bf5", linewidth=2, markersize=5, label="Macro F1")
ax2.plot(epochs, history["val_accuracy"], "s--", color="#34a853", linewidth=2, markersize=5, label="Accuracy")
best_ep = history["epoch"][history["val_macro_f1"].index(max(history["val_macro_f1"]))]
ax2.axvline(x=best_ep, color="gray", linestyle=":", linewidth=1.5, label=f"Best epoch ({best_ep})")
ax2.set_title("Val Macro F1 & Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Score")
ax2.set_ylim(0, 1)
ax2.set_xticks(epochs)
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# Plot 3: Per-Class F1 over Epochs
ax3 = fig.add_subplot(gs[1, 0])
class_colors = {
    "no_reason": "#34a853", "bias_or_abuse": "#e05c5c",
    "jobs_and_careers": "#fbbc04", "medication": "#9b59b6",
    "relationship": "#4c8bf5", "alienation": "#ff7f0e",
}
for cls, color in class_colors.items():
    ax3.plot(epochs, history[f"val_{cls}_f1"], "o-", color=color,
             linewidth=2, markersize=5, label=cls.replace("_", " ").title())
ax3.axhline(y=0.40, color="red", linestyle=":", linewidth=1, alpha=0.6, label="Bias/abuse warn (0.40)")
ax3.set_title("Per-Class F1 over Epochs")
ax3.set_xlabel("Epoch")
ax3.set_ylabel("F1 Score")
ax3.set_ylim(0, 1)
ax3.set_xticks(epochs)
ax3.legend(fontsize=7, ncol=2)
ax3.grid(True, alpha=0.3)

# Plot 4: Final Test Confusion Matrix
ax4 = fig.add_subplot(gs[1, 1])
cm = test_metrics["confusion_matrix"]
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
im = ax4.imshow(cm_norm, interpolation="nearest", cmap="Blues", vmin=0, vmax=1)
plt.colorbar(im, ax=ax4, fraction=0.046, pad=0.04)
ax4.set_xticks(range(len(LABEL_NAMES)))
ax4.set_yticks(range(len(LABEL_NAMES)))
ax4.set_xticklabels(LABEL_NAMES, rotation=35, ha="right", fontsize=8)
ax4.set_yticklabels(LABEL_NAMES, fontsize=8)
ax4.set_title("Test Confusion Matrix (Normalized)")
ax4.set_xlabel("Predicted")
ax4.set_ylabel("True")
for i in range(len(LABEL_NAMES)):
    for j in range(len(LABEL_NAMES)):
        val = cm_norm[i, j]
        ax4.text(j, i, f"{val:.2f}", ha="center", va="center",
                 fontsize=7, color="white" if val > 0.6 else "black")

plt.savefig(os.path.join(CHECKPOINT_DIR, "head2_training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to checkpoints/head2_training_curves.png")

In [ ]:
print("=" * 60)
print("OUTPUT FILES")
print("=" * 60)

for fname in sorted(os.listdir(CHECKPOINT_DIR)):
    fpath = os.path.join(CHECKPOINT_DIR, fname)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  {fname}  ({size_mb:.2f} MB)")

print(f"\nAll done. Download from the Output tab on Kaggle.")